# 03 — Artificial Neuron, Linear Algebra, and ANN Architecture (Manual NumPy)


> **Learning contract.** Every code cell is preceded by an explanation of what the code does, why the operation exists mathematically, what tensor/array shapes are expected, and what production or business failure it prevents. Run the notebooks in numerical order in a fresh Conda environment.


A neuron computes an affine transformation followed by a nonlinearity. For one neuron, $z=\mathbf{w}^T\mathbf{x}+b$. For a batch and a layer, $Z=XW+b$. The matrix form is not an optimization trick—it is the actual mathematical object a dense layer represents.

Our primary network is **784 → 64 → 10**. Therefore $W_1\in\mathbb{R}^{784\times64}$, $b_1\in\mathbb{R}^{64}$, $W_2\in\mathbb{R}^{64\times10}$, $b_2\in\mathbb{R}^{10}$.


## Code walkthrough — calculate one neuron by hand
We deliberately use four pixel-like inputs so every multiplication is visible. `contributions = x * w` exposes which features push the neuron upward or downward; the bias shifts the activation threshold independently of the current input.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
x = np.array([0.8, 0.1, 0.6, 0.0])
w = np.array([0.5, -0.3, 0.2, 0.7])
b = -0.15
contributions = x * w
z = contributions.sum() + b
print('x*w =', contributions)
print('sum(x*w)=', contributions.sum(), 'bias=', b, 'z=', z)
plt.bar(['x1w1','x2w2','x3w3','x4w4','bias'], [*contributions,b])
plt.axhline(0, linewidth=1); plt.title('Visible contributors to one neuron pre-activation'); plt.show()


## Code walkthrough — instantiate the full Manual NumPy architecture
The framework-specific model below still represents the same two affine transformations. Compare its parameter shapes against the manual matrix dimensions. The total parameter count is $784\times64+64+64\times10+10=50,890$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import urllib.request
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'
ARTIFACT_DIR = ROOT / 'artifacts'
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)
MNIST_PATH = DATA_DIR / 'mnist.npz'
MNIST_URL = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'


def load_official_mnist():
    if not MNIST_PATH.exists():
        print('Downloading official MNIST archive to', MNIST_PATH)
        urllib.request.urlretrieve(MNIST_URL, MNIST_PATH)
    with np.load(MNIST_PATH) as data:
        return data['x_train'], data['y_train'], data['x_test'], data['y_test']


def balanced_subset(x, y, per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    selected = []
    for cls in range(10):
        candidates = np.flatnonzero(y == cls)
        selected.extend(rng.choice(candidates, size=per_class, replace=False))
    selected = np.asarray(selected)
    rng.shuffle(selected)
    return x[selected], y[selected]


def prepare_splits():
    x_train_raw, y_train_raw, x_test_raw, y_test_raw = load_official_mnist()
    x_dev, y_dev = balanced_subset(x_train_raw, y_train_raw, per_class=600)
    x_test, y_test = balanced_subset(x_test_raw, y_test_raw, per_class=100, seed=SEED + 1)
    x_train, x_val, y_train, y_val = train_test_split(
        x_dev,
        y_dev,
        test_size=1000,
        random_state=SEED,
        stratify=y_dev,
    )
    def transform(x):
        return x.reshape(len(x), -1).astype('float32') / 255.0
    return transform(x_train), y_train, transform(x_val), y_val, transform(x_test), y_test
rng = np.random.default_rng(SEED)
W1 = rng.normal(0, np.sqrt(2/784), size=(784, 64)).astype('float32')
b1 = np.zeros((1, 64), dtype='float32')
W2 = rng.normal(0, np.sqrt(2/64), size=(64, 10)).astype('float32')
b2 = np.zeros((1, 10), dtype='float32')

def relu(z): return np.maximum(z, 0)
def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)
def forward(X):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    logits = a1 @ W2 + b2
    return z1, a1, logits

print('W1', W1.shape, 'b1', b1.shape, 'W2', W2.shape, 'b2', b2.shape)
print('parameters', W1.size+b1.size+W2.size+b2.size)


## Engineering inference
Width increases representational capacity but also memory, FLOPs and overfitting risk. Architecture is therefore both a statistical decision and a systems decision. Parameter count is the first bridge between mathematics and deployment cost.
